In [25]:
# function to return counts, channels, and time from .mca files
import matplotlib.pyplot as plt
import numpy as np
import math as m
import glob
from pathlib import Path

def mca_extraction(filename):
    '''
    Function reads an .mca file and returns lists corresponding
    to counts, channels, and time.
    ______________
    Parameters
    - .mca file: file containing gamma ray detector data.
    
    ______________
    Outputs
    - Counts: List of floats values
        
    - Channels: List of float values
        
    - Time: float value
        
    '''

    with open(filename, 'r') as file:
        spectrum_data = file.readlines()

    counts = []
    time = float(spectrum_data[8].replace(' ', '').replace('REAL_TIME-', ''))
    
    beginning = spectrum_data.index("<<DATA>>\n") + 1
    
    for line in spectrum_data[beginning:]:

        if line == "<<END>>\n":

            break
        else:
            counts.append(float(line))

    channels = list(range(len(counts)))
        
    return counts, channels, time


In [26]:
def extraction(filename):
    with open(filename, 'r') as file:
        spectrum_data = file.readlines()

    counts = []
    is_data_reached = False
    for line in spectrum_data:
        line = line.strip()        #It'll remove next line jump in readline
        lines = [line.replace(' ', '') for line in line]
           #ndex =np.where(spectrum_data == "$MEAS_TIM:")
           #print(spectrum_data)
        index = spectrum_data.index("$MEAS_TIM:\n")
        time = spectrum_data[index+1]
        time = float(time.split(" ")[0])

        if line == "$DATA:":
            is_data_reached = True   #this will tell the loop do operations after Data line is reached 
            continue
        if line == "$ROI:":
            break
        if is_data_reached:
            if line == "0 1023":      
                continue
            counts.append(float(line.split(",")[0]))
        channels = list(range(len(counts)))
    return counts, channels, time

In [27]:
def file_loader():

    here = Path('.')

    spe_files = here.rglob('*.Spe')
    mca_files = here.rglob('*.mca')
    
    return spe_files, mca_files
    

In [28]:
spefiles, mcafiles = file_loader()

for file in spefiles:
    print(file)

print('_____________________')

for file in mcafiles:
    print(file)


CUITS\BGO_Am_0_10cm_180s.Spe
CUITS\BGO_Am_0_20cm_180s.Spe
CUITS\BGO_Am_45(1)_10cm_180s.Spe
CUITS\BGO_Am_45(2)_10cm_180s.Spe
CUITS\BGO_Am_90(1)_10cm_180s.Spe
CUITS\BGO_Am_90(2)_10cm_180s.Spe
CUITS\BGO_Background_180s.Spe
CUITS\BGO_Background_300s.Spe
CUITS\BGO_Ba_0_10.Spe
CUITS\BGO_Ba_0_10cm_180s.Spe
CUITS\BGO_Ba_0_10cm_300s.Spe
CUITS\BGO_Ba_0_20cm_300s.Spe
CUITS\BGO_Ba_30(1)_10cm_300s.Spe
CUITS\BGO_Ba_30(2)_10cm_300s.Spe
CUITS\BGO_Ba_45(1)_10cm_300s.Spe
CUITS\BGO_Ba_45(2)_10cm_300s.Spe
CUITS\BGO_Ba_90(1)_10cm_300s.Spe
CUITS\BGO_Ba_90(2)_10cm_300s.Spe
CUITS\BGO_Co_0_10cm_420s.Spe
CUITS\BGO_Co_0_20cm_420s.Spe
CUITS\BGO_Co_45(1)_10cm_420s.Spe
CUITS\BGO_Co_45(2)_10cm_420s.Spe
CUITS\BGO_Co_90(1)_10cm_420s.Spe
CUITS\BGO_Co_90(2)_10cm_420s.Spe
CUITS\BGO_Cs_0_10cm.Spe
CUITS\BGO_Cs_0_10cm_180s.Spe
CUITS\BGO_Cs_0_20cm.Spe
CUITS\BGO_Cs_0_20cm_180s.Spe
CUITS\BGO_Cs_45(1)_10cm.Spe
CUITS\BGO_Cs_45(1)_10cm_180s.Spe
CUITS\BGO_Cs_45(2)_10cm.Spe
CUITS\BGO_Cs_45(2)_10cm_180s.Spe
CUITS\BGO_Cs_90(1)_10cm.S

In [46]:

spe_file = Path("CUITS/BGO_Co_0_10cm_420s.Spe")


with open(spe_file, 'r') as file:
    spect_data = file.readlines()

#print(spect_data)

#for line in spect_data:
    #print(line.strip())

In [47]:
def get_time_spe(filename):
    """Read a .Spe file. Returns date measured."""

    source = str(filename).split('_')[1]
    
    with open(filename, "r") as f:
        lines = [line.strip() for line in f]

    # $DATE_MEA: is followed by a line like "10/12/2026 16:18:58" 
    date = lines[lines.index('$DATE_MEA:') + 1]

    
    
    return date, source

timee, sourcee = get_time_spe(spe_file)
print(timee)
print(sourcee)

10/13/2026 12:22:59
Co


In [52]:
mca_file = Path("CUITS/CdTe_Co_0_10cm_420s.mca")


with open(mca_file, 'r') as file:
    spect_data = file.readlines()

#print(spect_data)

#for line in spect_data:
    #print(line.strip())

In [53]:
def eff_info_mca(filename):
    """
    Function that reads an .mca file and returns date measured and the type of source.
    
    **NOTE** There is no information in the files that states what radio-isotope
    the file belongs to, other than the file naming convention used when
    collecting data. For this reason this function relys on that fact when returning
    the radio-isotope.
    """
    radio_isotope = str(filename).split('_')[1]

    with open(filename, 'r') as file:
        spectrum_data = file.readlines()

    meas_date = spectrum_data[9].replace('START_TIME - ', '').strip()
    
    return meas_date, radio_isotope

datee, isotope = eff_info_mca(mca_file)

print(datee)
print(isotope)

10/20/2026 12:45:08
Co


In [54]:
from datetime import datetime

def source_age(meas_date):
    '''
    Function that calculates the age of the source from the time that the
    source was calibrated and the time it was measured.
    '''
    
    # all measurements taken from upstairs sources.  
    calibration_date = '1979/02/01 12:00:00'

    cal_time = datetime.strptime(calibration_date, "%Y/%m/%d %H:%M:%S")
    meas_time = datetime.strptime(meas_date, "%m/%d/%Y %H:%M:%S")

    age_of_source = (meas_time - cal_time).total_seconds()
    
    return age_of_source

age = source_age(datee)
print(age)

1505781908.0


In [55]:
def source_activity(file_date, source):
    '''
    Function that calculates the remaining activity of a radioisotope source 
    by applying its halflife and total time to its initial activity. 

    is dependant on the source type where the function chooses
    which hardcoded source values to use.
    __________________
    Parameters
        init_activity: Float
        The initial activity of the radio-isotope from when it was first calibrated.
        
    ___________________
    Outputs
        
    '''
    #make this into a dictinary where the source is the key
    source_params = {
        "sources": ['Am','Ba','Co','Cs'],
        "initial_activity": [11.92, 10.82, 11.32, 12.41],
        "accuracy": [5.0, 4.8, 1.9, 3.7],
        "HL3_confirmed": [432.2, 10.537, 5.2714, 30.09],
        "emission_frac": [0.999,0.999,0.999,0.999]
    }

    #here index 0 is Am, then Ba, then Co, then Cs
    #init_activity = [11.92, 10.82, 11.32, 12.41] # uCi, initial activities of sources back in 1979
    #accuracy = [5.0, 4.8, 1.9, 3.7] # included for error propigation
    #HL3_confirmed = [432.2, 10.537, 5.2714, 30.09] # half lifes of sources
    #emission_frac = [,,,] # emission fractions of sources

    # index corresponding to source
    i = source_params["sources"].index(source)
    
    # getting corresponding values within dictionary 
    init_activity = source_params["initial_activity"][i]
    accuracy = source_params["accuracy"][i]
    half_life = source_params["HL3_confirmed"][i]
    emission_frac = source_params["emission_frac"][i]

    # converting half_life to seconds considering leap years
    HL_seconds = half_life * 365.25 * 24 * 60 * 60 
    
    time = source_age(file_date)
    remaining_activity = init_activity * (0.5 **(time/HL_seconds))
    activity = remaining_activity * emission_frac

    return activity

act = source_activity(datee, isotope)

print(act)

0.02130915640526524
